# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 01.01 · Adquisición incremental de subtítulos

Reutiliza transcripciones canónicas y cachés por `video_id`; solo consulta YouTube para candidatos nuevos y nunca descarga audio o video.

La adquisición nueva usa `yt-dlp` para localizar pistas de subtítulos [1]. Las transcripciones automáticas se conservan como insumo imperfecto, no como verdad textual, porque se han documentado sesgos de dialecto y género en el subtitulado automático de YouTube [2]. Toda ampliación debe respetar los términos de la plataforma [3] y la evaluación ética contextual recomendada para investigación en Internet [4]. La reutilización de cachés y la selección de candidatos son decisiones locales registradas en manifiestos.

**Contrato v2.1:** `SEGURO` + cuatro daños entrenados, incluida `ATAQUE_POR_GENERO_IDENTIDAD`. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Proyecto:', ROOT)


## Preflight

In [ ]:
from moderacion_peru.artifacts import artifact_status
artifact_status(ROOT)

## Reutilización de snapshots existentes

In [ ]:
from moderacion_peru.acquisition import (bootstrap_canonical_from_existing, discover_existing_transcript_sources, fetch_youtube_subtitles, ingest_incremental, load_candidates)
CANONICAL = ROOT/'datos/raw/transcripts_raw.jsonl'
CACHE = ROOT/'datos/raw/transcripts_cache'
sources = discover_existing_transcript_sources(ROOT, canonical_path=CANONICAL)
reuse_stats = bootstrap_canonical_from_existing(sources, CANONICAL)
print(reuse_stats)

## Candidatos y caché

In [ ]:
CANDIDATE_FILES = [ROOT/'datos/raw/video_candidates.jsonl', ROOT/'datos/raw/videos_candidatos.csv']
candidates_by_id = {}
for source in CANDIDATE_FILES:
    for row in load_candidates(source):
        candidates_by_id.setdefault(str(row['video_id']), row)
candidates = list(candidates_by_id.values())
print('Candidatos:', len(candidates), '· los ya existentes se omitirán')

## Ejecución controlada

In [ ]:
FETCH_NEW = False  # Cambie a True solo para consultar candidatos no vistos
if candidates:
    stats = ingest_incremental(candidates, CANONICAL, CACHE, fetcher=fetch_youtube_subtitles if FETCH_NEW else None)
    print(stats)
else:
    print('Agregue candidatos con video_id y url; el corpus existente no se vuelve a descargar.')

## Referencias

[1] yt-dlp contributors, "yt-dlp: A Feature-Rich Command-Line Audio/Video Downloader," GitHub repository, 2026. [Online]. Available: https://github.com/yt-dlp/yt-dlp. Accessed: Aug. 5, 2026.

[2] R. Tatman, "Gender and Dialect Bias in YouTube's Automatic Captions," in Proc. 1st ACL Workshop Ethics NLP, 2017, pp. 53–59, doi: 10.18653/v1/W17-1606.

[3] YouTube, "Terms of Service," Nov. 2023. [Online]. Available: https://www.youtube.com/t/terms. Accessed: Aug. 5, 2026.

[4] A. S. franzke, A. Bechmann, M. Zimmer, et al., "Internet Research: Ethical Guidelines 3.0," Association of Internet Researchers, 2020. [Online]. Available: https://aoir.org/reports/ethics3.pdf